# Notebook 04 — Neural-Operator Deep Smoothing, Full CPU Research Run

**Goal.** NB02 and NB03 fit one surface at a time. This notebook trains **operators**: one model learns the map from an irregular cloud of option quotes to a smooth implied-volatility surface, then applies to unseen days without per-day retraining.

This version is built for a **serious CPU run**:

- it uses **all available daily surfaces** by default;
- an epoch means a **full pass through the relevant surface bank**, not one gradient step;
- it trains both executable operator families:
  - **DeepONet / DeepSets**: fast, stable, permutation-invariant baseline;
  - **modified GNO**: closer to the paper, heavier, CPU-practicable through point subsampling;
- it runs the low-data transfer experiment:

| Regime | Pre-training | Fine-tuning | Interpretation |
|---|---|---|---|
| **R1 real-only** | none | real train days | academic daily-data baseline |
| **R2 synth-only** | synthetic SSVI bank | none | zero-shot synthetic-to-real transfer |
| **R3 pretrain + finetune** | synthetic SSVI bank | real train days | whether synthetic pre-training closes the data gap |

**Important CPU note.** A full GNO over every quote of every day is not CPU-realistic. The notebook therefore uses all days, but samples a fresh subset of quotes per surface at every epoch. This is not a shortcut; it is the same input-subsampling augmentation principle used by operator methods, and over epochs it exposes the model to the whole dataset.

The output is analysis-ready: metrics, arbitrage audits, invariance diagnostics, train curves, per-day predictions, and comparison tables are written to `data/clean/nb04_*`.

In [1]:
import os
import json
import time
import copy
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

torch.set_default_dtype(torch.float32)
np.set_printoptions(precision=4, suppress=True)
pio.renderers.default = os.environ.get("PLOTLY_RENDERER", "notebook_connected")


def default_clean_dir():
    env = os.environ.get("THESIS_OUT_DIR")
    if env:
        return Path(env)
    for candidate in [Path("data/clean"), Path("main/data/clean")]:
        if (candidate / "option_prices_clean.parquet").exists():
            return candidate
    return Path("data/clean")


OUT_DIR = default_clean_dir()
OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUT_DIR / "nb04_operator_run"
RUN_DIR.mkdir(parents=True, exist_ok=True)
REAL_PARQUET = Path(os.environ.get("THESIS_OPT_PARQUET", str(OUT_DIR / "option_prices_clean.parquet")))

SEED = int(os.environ.get("NB04_SEED", "0"))
DEVICE = torch.device(os.environ.get("NB04_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))

# Defaults are intentionally serious for CPU. Increase for final overnight runs.
N_SYNTH = int(os.environ.get("NB04_N_SYNTH", "1000"))
LIMIT_DAYS_RAW = os.environ.get("NB04_LIMIT_DAYS", "None")
LIMIT_DAYS = None if LIMIT_DAYS_RAW in ("", "None", "none", "ALL", "all") else int(LIMIT_DAYS_RAW)

EPOCHS_PRE = int(os.environ.get("NB04_EPOCHS_PRE", "25"))
EPOCHS_R1 = int(os.environ.get("NB04_EPOCHS_R1", "25"))
EPOCHS_FT = int(os.environ.get("NB04_EPOCHS_FT", "12"))

MAX_QUOTES_LOAD = int(os.environ.get("NB04_MAX_QUOTES_PER_DAY", "2500"))
FIT_POINTS_DEEP = int(os.environ.get("NB04_FIT_POINTS_DEEP", "512"))
INPUT_POINTS_DEEP = int(os.environ.get("NB04_INPUT_POINTS_DEEP", "900"))
FIT_POINTS_GNO = int(os.environ.get("NB04_FIT_POINTS_GNO", "160"))
INPUT_POINTS_GNO = int(os.environ.get("NB04_INPUT_POINTS_GNO", "220"))

BATCH_DEEP = int(os.environ.get("NB04_BATCH_DEEP", "8"))
BATCH_GNO = int(os.environ.get("NB04_BATCH_GNO", "1"))

LATENT = int(os.environ.get("NB04_LATENT", "64"))
ENC_H = int(os.environ.get("NB04_ENC_H", "128"))
TRUNK_H = int(os.environ.get("NB04_TRUNK_H", "128"))
GNO_CHANNELS = int(os.environ.get("NB04_GNO_CHANNELS", "8"))
GNO_LAYERS = int(os.environ.get("NB04_GNO_LAYERS", "2"))
GNO_K = int(os.environ.get("NB04_GNO_K", "16"))
GNO_RHO = float(os.environ.get("NB04_GNO_RHO", "0.35"))

GRID_NK = int(os.environ.get("NB04_GRID_NK", "22"))
GRID_NT = int(os.environ.get("NB04_GRID_NT", "9"))
EPS_BUT = 1e-3
LAMBDAS = dict(fit=1.0, but=10.0, cal=10.0, reg_t=0.01, reg_k=0.01)

torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

cfg = {k: v for k, v in list(globals().items()) if k.startswith("NB04_")}
print("REAL_PARQUET:", REAL_PARQUET.resolve())
print("OUT_DIR     :", OUT_DIR.resolve())
print("RUN_DIR     :", RUN_DIR.resolve())
print("DEVICE      :", DEVICE)
print("all days    :", LIMIT_DAYS is None)
print("epochs      :", dict(pretrain=EPOCHS_PRE, real=EPOCHS_R1, finetune=EPOCHS_FT))

REAL_PARQUET: /Users/hamzaelarji/Desktop/All/Bureau/cours/Imperial College/Ze master thesis /main/data/clean/option_prices_clean.parquet
OUT_DIR     : /Users/hamzaelarji/Desktop/All/Bureau/cours/Imperial College/Ze master thesis /main/data/clean
RUN_DIR     : /Users/hamzaelarji/Desktop/All/Bureau/cours/Imperial College/Ze master thesis /main/data/clean/nb04_operator_run
DEVICE      : cpu
all days    : True
epochs      : {'pretrain': 25, 'real': 25, 'finetune': 12}


## 1. Data: real daily bank and synthetic SSVI bank

A surface is a variable-size set

$$
\{(k_i,\tau_i,\sigma_i)\}_{i=1}^{p}.
$$

The real bank comes from NB01's `option_prices_clean.parquet`, filtered to OTM quotes with valid IV and valid discount. The split is chronological: earliest dates train, middle dates validation, latest dates test.

The synthetic bank is generated from SSVI with realistic random parameters and irregular sparse quote layouts. It is used only for pre-training and zero-shot transfer.

In [2]:
K_LO, K_HI = -0.50, 0.35
T_LO, T_HI = 0.05, 1.50


def ssvi_total_variance(k, theta, rho, eta, gamma):
    phi = eta * theta ** (-gamma)
    return 0.5 * theta * (1 + rho * phi * k + np.sqrt((phi * k + rho) ** 2 + (1 - rho ** 2)))


def sample_ssvi_params(r):
    return dict(
        rho=float(r.uniform(-0.80, -0.10)),
        eta=float(r.uniform(0.40, 1.45)),
        gamma=float(r.uniform(0.18, 0.50)),
        alpha=float(r.uniform(0.015, 0.12)),
        beta=float(r.uniform(0.75, 1.20)),
    )


def synth_surface(r, params=None, n_slices=(5, 10), n_per=(8, 24), noise=0.012):
    p = params or sample_ssvi_params(r)
    taus = np.sort(r.uniform(T_LO, T_HI, r.integers(n_slices[0], n_slices[1] + 1)))
    ks, ts, ivs = [], [], []
    for tau in taus:
        n = int(r.integers(n_per[0], n_per[1] + 1))
        k = np.sort(r.uniform(K_LO, K_HI, n))
        theta = p["alpha"] * tau ** p["beta"]
        w = ssvi_total_variance(k, theta, p["rho"], p["eta"], p["gamma"])
        iv = np.sqrt(np.maximum(w / tau, 1e-10)) * (1 + noise * r.standard_normal(n))
        ks.append(k); ts.append(np.full(n, tau)); ivs.append(iv)
    return dict(date="synthetic", k=np.concatenate(ks), tau=np.concatenate(ts), iv=np.concatenate(ivs), kind="synthetic")


def bounded_sample(k, tau, iv, max_n, r):
    ok = (
        np.isfinite(k) & np.isfinite(tau) & np.isfinite(iv)
        & (k >= K_LO) & (k <= K_HI)
        & (tau >= T_LO) & (tau <= T_HI)
        & (iv > 0) & (iv < 5)
    )
    k, tau, iv = k[ok], tau[ok], iv[ok]
    if len(k) > max_n:
        idx = r.choice(len(k), size=max_n, replace=False)
        k, tau, iv = k[idx], tau[idx], iv[idx]
    order = np.lexsort((k, tau))
    return k[order], tau[order], iv[order]


def load_real_bank():
    assert REAL_PARQUET.exists(), f"Missing clean parquet: {REAL_PARQUET}. Run NB01 first."
    cols = pl.scan_parquet(REAL_PARQUET).collect_schema().names()
    iv_col = "iv_om" if "iv_om" in cols else "impl_volatility"
    filters = (
        pl.col("is_otm")
        & pl.col(iv_col).is_not_null()
        & pl.col("k").is_not_null()
        & pl.col("tau").is_not_null()
    )
    if "discount_valid" in cols:
        filters = filters & pl.col("discount_valid")
    scan = pl.scan_parquet(REAL_PARQUET).filter(filters)
    dates = scan.select("date").unique().collect(engine="streaming")["date"].sort().to_list()
    if LIMIT_DAYS is not None:
        dates = dates[-LIMIT_DAYS:]
    df = (
        scan.filter(pl.col("date").is_in(dates))
        .select(["date", "tau", "k", iv_col])
        .rename({iv_col: "iv"})
        .collect(engine="streaming")
        .sort("date")
    )

    r = np.random.default_rng(SEED + 17)
    days = []
    for d in dates:
        sub = df.filter(pl.col("date") == d)
        k, tau, iv = bounded_sample(
            sub["k"].to_numpy(),
            sub["tau"].to_numpy(),
            sub["iv"].to_numpy(),
            MAX_QUOTES_LOAD,
            r,
        )
        if len(k) >= 40:
            days.append(dict(date=str(d), k=k, tau=tau, iv=iv, kind="real"))
    return days


real_bank = load_real_bank()
synth_bank = [synth_surface(rng) for _ in range(N_SYNTH)]

n = len(real_bank)
i_tr, i_va = int(0.60 * n), int(0.80 * n)
train_days, val_days, test_days = real_bank[:i_tr], real_bank[i_tr:i_va], real_bank[i_va:]

def size_summary(bank):
    sizes = np.array([len(s["k"]) for s in bank])
    return dict(n=len(bank), min=int(sizes.min()), median=int(np.median(sizes)), max=int(sizes.max()))

print("real bank      :", size_summary(real_bank), real_bank[0]["date"], "->", real_bank[-1]["date"])
print("train/val/test :", len(train_days), len(val_days), len(test_days))
print("synthetic bank :", size_summary(synth_bank))
assert train_days and val_days and test_days, "Chronological split produced an empty subset."

real bank      : {'n': 1926, 'min': 2367, 'median': 2500, 'max': 2500} 2018-01-02 -> 2025-08-29
train/val/test : 1155 385 386
synthetic bank : {'n': 1000, 'min': 48, 'median': 117, 'max': 201}


## 2. Shared numerical tools

All models are trained in IV space but audited in total variance $w=\sigma^2\tau$. The butterfly check uses the Durrleman function computed by finite differences on a fixed grid, matching the architecture-agnostic derivative treatment in the operator paper.

In [3]:
def as_t(x):
    return torch.as_tensor(x, dtype=torch.float32, device=DEVICE)


def sample_idx(n, m, r):
    if m is None or n <= m:
        return np.arange(n)
    return np.sort(r.choice(n, size=m, replace=False))


def scale_coords_t(k, tau):
    k = (k - (K_LO + K_HI) / 2) / ((K_HI - K_LO) / 2)
    tau = (tau - (T_LO + T_HI) / 2) / ((T_HI - T_LO) / 2)
    return torch.stack([k, tau], dim=-1)


KG = np.linspace(K_LO + 0.02, K_HI - 0.02, GRID_NK)
TG = np.linspace(T_LO + 0.02, T_HI - 0.02, GRID_NT)
KKG, TTG = np.meshgrid(KG, TG)
KG_FLAT, TG_FLAT = KKG.ravel(), TTG.ravel()
DK = float(KG[1] - KG[0])
TTG_T = as_t(TTG)
KKG_T = as_t(KKG)


def durrleman_g_np(k, w, wk, wkk):
    return (1 - k * wk / (2 * w)) ** 2 - (wk ** 2 / 4) * (1 / w + 0.25) + wkk / 2


def durrleman_g_t(k, w, wk, wkk):
    return (1 - k * wk / (2 * w)) ** 2 - (wk ** 2 / 4) * (1 / w + 0.25) + wkk / 2


def vega_weights(k, tau, iv):
    d1 = (-k + 0.5 * iv ** 2 * tau) / (iv * torch.sqrt(tau))
    vega = torch.exp(-0.5 * d1 ** 2) / np.sqrt(2 * np.pi) * torch.sqrt(tau)
    return vega / torch.clamp(vega.mean(), min=1e-8)


def fd_audit_from_grid(V):
    W = V ** 2 * TTG
    Wk = (W[:, 2:] - W[:, :-2]) / (2 * DK)
    Wkk = (W[:, 2:] - 2 * W[:, 1:-1] + W[:, :-2]) / (DK ** 2)
    G = durrleman_g_np(KKG[:, 1:-1], W[:, 1:-1], Wk, Wkk)
    cal_viol = np.mean((W[1:, :] - W[:-1, :]) < -1e-8) * 100
    return dict(min_g=float(G.min()), butterfly_viol_pct=float(np.mean(G < -1e-8) * 100), calendar_viol_pct=float(cal_viol))


# Validate finite differences once against an SSVI closed form.
p0 = dict(rho=-0.5, eta=0.9, gamma=0.4)
theta0 = 0.04 * 0.5
phi0 = p0["eta"] * theta0 ** (-p0["gamma"])
w = ssvi_total_variance(KG, theta0, **p0)
sq = np.sqrt((phi0 * KG + p0["rho"]) ** 2 + 1 - p0["rho"] ** 2)
wp = 0.5 * theta0 * (p0["rho"] * phi0 + phi0 * (phi0 * KG + p0["rho"]) / sq)
wpp = 0.5 * theta0 * phi0 ** 2 * (1 - p0["rho"] ** 2) / sq ** 3
wk_fd = (w[2:] - w[:-2]) / (2 * DK)
wkk_fd = (w[2:] - 2 * w[1:-1] + w[:-2]) / (DK ** 2)
err = np.max(np.abs(durrleman_g_np(KG[1:-1], w[1:-1], wk_fd, wkk_fd) - durrleman_g_np(KG, w, wp, wpp)[1:-1]))
print(f"FD Durrleman validation max error: {err:.2e}")
assert err < 3e-2, "Finite-difference grid is too coarse; increase NB04_GRID_NK."

FD Durrleman validation max error: 3.64e-03


## 3. Models

Two operator models are trained with the same protocol and metrics.

**DeepONet / DeepSets.** Fast CPU baseline: mean-pooled quote encoder plus coordinate trunk.

**Modified GNO.** Paper-style graph neural operator: non-local first layer and maturity-truncated neighborhoods. On CPU it is trained with smaller channels/layers and quote subsampling.

In [4]:
class MLP(nn.Module):
    def __init__(self, sizes, activation=nn.Tanh):
        super().__init__()
        layers = []
        for i, (a, b) in enumerate(zip(sizes[:-1], sizes[1:])):
            layers.append(nn.Linear(a, b))
            if i < len(sizes) - 2:
                layers.append(activation())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class DeepSetONet(nn.Module):
    model_name = "deeponet"
    fit_points = FIT_POINTS_DEEP
    input_points = INPUT_POINTS_DEEP
    batch_size = BATCH_DEEP

    def __init__(self):
        super().__init__()
        self.encoder = MLP([3, ENC_H, ENC_H, LATENT])
        self.psi = MLP([LATENT, ENC_H, LATENT])
        self.trunk = MLP([2, TRUNK_H, TRUNK_H, LATENT])
        self.bias = nn.Parameter(torch.zeros(()))

    def encode_surface(self, surf, input_idx):
        k = as_t(surf["k"][input_idx])
        tau = as_t(surf["tau"][input_idx])
        iv = as_t(surf["iv"][input_idx])
        x = torch.cat([scale_coords_t(k, tau), 5.0 * iv[:, None]], dim=-1)
        return self.encoder(x).mean(dim=0)

    def predict_surface(self, surf, kq, tq, input_idx):
        z = self.encode_surface(surf, input_idx)
        coords = scale_coords_t(as_t(kq), as_t(tq))
        c = self.psi(z[None, :])[0]
        t = self.trunk(coords)
        return F.softplus((t * c).sum(dim=-1) + self.bias) + 1e-6

In [5]:
def ffn(sizes, activation=nn.GELU):
    layers = []
    for i, (a, b) in enumerate(zip(sizes[:-1], sizes[1:])):
        layers.append(nn.Linear(a, b))
        if i < len(sizes) - 2:
            layers.append(activation())
    return nn.Sequential(*layers)


def build_neighbors(x_in, y_out, rho_bar=GNO_RHO, K=GNO_K):
    nbrs = []
    for y in y_out:
        mask = (x_in[:, 1] - y[1]).abs() <= rho_bar
        idx = torch.nonzero(mask, as_tuple=False).flatten()
        if idx.numel() == 0:
            idx = torch.arange(len(x_in), device=x_in.device)
        d = ((x_in[idx] - y) ** 2).sum(-1)
        idx = idx[d.argsort()]
        stride = max(1, int(np.ceil(idx.numel() / K)))
        nbrs.append(idx[::stride][:K])
    return nbrs


class GNOLayer(nn.Module):
    def __init__(self, c_in, c_out, local=True, hidden=64):
        super().__init__()
        self.P = ffn([c_in, hidden, c_in])
        self.kW = ffn([4 + c_in + 1, hidden, hidden, c_out * c_in])
        self.kb = ffn([4 + c_in + 1, hidden, hidden, c_out])
        self.W = nn.Linear(c_in, c_out) if local else None
        self.Q = ffn([c_out, hidden, c_out])
        self.act = nn.GELU()
        self.c_in, self.c_out = c_in, c_out

    def forward(self, h_in, x_in, y_out, v_in, nbrs, h_self=None):
        h_t = self.P(h_in)
        out = []
        for j, y in enumerate(y_out):
            idx = nbrs[j]
            xz, hz, vz = x_in[idx], h_t[idx], v_in[idx]
            feat = torch.cat([y.expand(len(idx), 2), xz, hz, vz], dim=-1)
            Wk = self.kW(feat).view(len(idx), self.c_out, self.c_in)
            bk = self.kb(feat)
            out.append((torch.einsum("nij,nj->ni", Wk, hz) + bk).mean(dim=0))
        out = torch.stack(out)
        if self.W is not None and h_self is not None:
            out = out + self.W(self.P(h_self))
        return self.act(self.Q(out))


class InterpGNO(nn.Module):
    model_name = "gno"
    fit_points = FIT_POINTS_GNO
    input_points = INPUT_POINTS_GNO
    batch_size = BATCH_GNO

    def __init__(self, channels=GNO_CHANNELS, J=GNO_LAYERS):
        super().__init__()
        self.lift = ffn([1, 64, channels])
        self.layers = nn.ModuleList([GNOLayer(channels, channels, local=(j > 0)) for j in range(J)])
        self.out = nn.Sequential(nn.Linear(channels, 1), nn.Softplus())

    def forward_coords(self, x_in, v_in, y_out):
        nbrs = build_neighbors(x_in, y_out)
        h_in = self.lift(v_in)
        h = self.layers[0](h_in, x_in, y_out, v_in, nbrs)
        nbrs_out = build_neighbors(y_out, y_out)
        v_zero = torch.zeros(len(y_out), 1, device=y_out.device)
        for layer in self.layers[1:]:
            h = layer(h, y_out, y_out, v_zero, nbrs_out, h_self=h)
        return self.out(h).squeeze(-1) + 1e-6

    def predict_surface(self, surf, kq, tq, input_idx):
        kin = as_t(surf["k"][input_idx])
        tin = as_t(surf["tau"][input_idx])
        vin = as_t(surf["iv"][input_idx])[:, None]
        x_in = scale_coords_t(kin, tin)
        y_out = scale_coords_t(as_t(kq), as_t(tq))
        return self.forward_coords(x_in, vin, y_out)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("DeepONet params:", count_params(DeepSetONet()))
print("GNO params     :", count_params(InterpGNO()))

DeepONet params: 67009
GNO params     : 34697


## 4. Loss, training loop, and evaluation

The training loop is deliberately written so that **one epoch = one pass through the whole bank**. For each surface, a fresh subset of input quotes and fit quotes is sampled. This makes CPU training feasible while still using every date and, across epochs, broadly all quotes.

In [6]:
def loss_terms(model, surf, r):
    n = len(surf["k"])
    input_idx = sample_idx(n, model.input_points, r)
    fit_idx = sample_idx(n, model.fit_points, r)

    k_fit = surf["k"][fit_idx]
    t_fit = surf["tau"][fit_idx]
    iv_fit_np = surf["iv"][fit_idx]
    iv_fit = as_t(iv_fit_np)
    pred = model.predict_surface(surf, k_fit, t_fit, input_idx)

    vw = vega_weights(as_t(k_fit), as_t(t_fit), iv_fit)
    fit = torch.sqrt(torch.mean(vw * ((pred - iv_fit) / torch.clamp(iv_fit, min=1e-6)) ** 2))

    V = model.predict_surface(surf, KG_FLAT, TG_FLAT, input_idx).reshape(GRID_NT, GRID_NK)
    W = V ** 2 * TTG_T
    Wk = (W[:, 2:] - W[:, :-2]) / (2 * DK)
    Wkk = (W[:, 2:] - 2 * W[:, 1:-1] + W[:, :-2]) / (DK ** 2)
    G = durrleman_g_t(KKG_T[:, 1:-1], W[:, 1:-1], Wk, Wkk)

    but = torch.relu(EPS_BUT - G).mean()
    cal = torch.relu(-(W[1:, :] - W[:-1, :])).mean()
    reg_t = torch.sqrt(torch.mean((V[2:, :] - 2 * V[1:-1, :] + V[:-2, :]) ** 2) + 1e-12)
    reg_k = torch.sqrt(torch.mean((V[:, 2:] - 2 * V[:, 1:-1] + V[:, :-2]) ** 2) + 1e-12)
    total = LAMBDAS["fit"] * fit + LAMBDAS["but"] * but + LAMBDAS["cal"] * cal + LAMBDAS["reg_t"] * reg_t + LAMBDAS["reg_k"] * reg_k
    return total, dict(fit=fit, but=but, cal=cal, reg_t=reg_t, reg_k=reg_k)


def batches(bank, batch_size, r):
    idx = r.permutation(len(bank))
    for start in range(0, len(idx), batch_size):
        yield [bank[i] for i in idx[start:start + batch_size]]


def train_full_pass(model, bank, epochs, lr, tag, eval_bank=None):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    r = np.random.default_rng(SEED + abs(hash(tag)) % 1_000_000)
    history = []
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        ep_losses = []
        ep_terms = []
        for batch in batches(bank, model.batch_size, r):
            opt.zero_grad(set_to_none=True)
            losses, term_rows = [], []
            for surf in batch:
                loss, terms = loss_terms(model, surf, r)
                losses.append(loss)
                term_rows.append({k: float(v.detach().cpu()) for k, v in terms.items()})
            batch_loss = torch.stack(losses).mean()
            batch_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            ep_losses.append(float(batch_loss.detach().cpu()))
            ep_terms.extend(term_rows)

        row = dict(model=model.model_name, tag=tag, epoch=ep, loss=float(np.mean(ep_losses)), seconds=time.time() - t0)
        for key in ["fit", "but", "cal", "reg_t", "reg_k"]:
            row[key] = float(np.mean([x[key] for x in ep_terms]))
        if eval_bank is not None and (ep == 1 or ep == epochs or ep % max(1, epochs // 5) == 0):
            ev = evaluate_model(model, eval_bank, max_days=min(20, len(eval_bank)), tag=f"{tag}_val")
            row.update({f"val_{k}": v for k, v in ev.items() if isinstance(v, (int, float))})
        history.append(row)

        if ep == 1 or ep == epochs or ep % max(1, epochs // 5) == 0:
            msg = f"[{tag}] epoch {ep:>3}/{epochs} full-pass loss={row['loss']:.4f}"
            if "val_rmse_volpts" in row:
                msg += f" | val rmse={row['val_rmse_volpts']:.3f} vol pts"
            msg += f" | elapsed={row['seconds']:.0f}s"
            print(msg)
    return model, history


@torch.no_grad()
def predict_np(model, surf, kq, tq, input_points=None):
    model.eval()
    n = len(surf["k"])
    # deterministic evaluation subset: evenly spaced after sorting
    m = input_points or model.input_points
    if n > m:
        idx = np.linspace(0, n - 1, m).round().astype(int)
    else:
        idx = np.arange(n)
    out = model.predict_surface(surf, np.asarray(kq), np.asarray(tq), idx)
    return out.detach().cpu().numpy()


@torch.no_grad()
def evaluate_model(model, days, max_days=None, tag="eval"):
    use_days = days if max_days is None else days[:max_days]
    rmses, maes, rels, min_gs, bviol, cviol = [], [], [], [], [], []
    for surf in use_days:
        pred = predict_np(model, surf, surf["k"], surf["tau"])
        err = pred - surf["iv"]
        rmses.append(float(np.sqrt(np.mean(err ** 2))) * 100)
        maes.append(float(np.mean(np.abs(err))) * 100)
        rels.append(float(np.sqrt(np.mean((err / surf["iv"]) ** 2))))
        V = predict_np(model, surf, KG_FLAT, TG_FLAT).reshape(GRID_NT, GRID_NK)
        audit = fd_audit_from_grid(V)
        min_gs.append(audit["min_g"])
        bviol.append(audit["butterfly_viol_pct"])
        cviol.append(audit["calendar_viol_pct"])
    return dict(
        tag=tag, n_days=len(use_days),
        rmse_volpts=float(np.mean(rmses)),
        rmse_sd_volpts=float(np.std(rmses)),
        mae_volpts=float(np.mean(maes)),
        rel_rmse=float(np.mean(rels)),
        min_g=float(np.min(min_gs)),
        butterfly_viol_pct=float(np.mean(bviol)),
        calendar_viol_pct=float(np.mean(cviol)),
    )

## 5. Main experiment: all models × R1/R2/R3

This is the heavy cell. On CPU, expect DeepONet to be moderate and GNO to be much slower. The outputs are checkpointed after each model/regime, so partial progress is still useful.

In [7]:
def save_history(rows, name):
    path = RUN_DIR / name
    pl.DataFrame(rows).write_parquet(path)
    print("written:", path)


def run_family(model_ctor, family_name, lr_pre, lr_real, lr_ft):
    histories = []
    results = []
    models = {}
    print("\n" + "=" * 80)
    print("MODEL FAMILY:", family_name)
    print("=" * 80)

    print("\nR2: synthetic-only")
    m_pre = model_ctor().to(DEVICE)
    print("params:", count_params(m_pre))
    m_pre, h = train_full_pass(m_pre, synth_bank, EPOCHS_PRE, lr_pre, f"{family_name}_R2_pretrain", eval_bank=val_days)
    histories += h
    models["R2 synth-only"] = copy.deepcopy(m_pre).cpu()
    res = evaluate_model(m_pre, test_days, tag=f"{family_name}_R2_synth_only")
    results.append(dict(model=family_name, regime="R2 synth-only", **res))

    print("\nR1: real-only")
    m_r1 = model_ctor().to(DEVICE)
    print("params:", count_params(m_r1))
    m_r1, h = train_full_pass(m_r1, train_days, EPOCHS_R1, lr_real, f"{family_name}_R1_real", eval_bank=val_days)
    histories += h
    models["R1 real-only"] = copy.deepcopy(m_r1).cpu()
    res = evaluate_model(m_r1, test_days, tag=f"{family_name}_R1_real_only")
    results.append(dict(model=family_name, regime="R1 real-only", **res))

    print("\nR3: synthetic pretrain + real finetune")
    m_r3 = copy.deepcopy(m_pre).to(DEVICE)
    m_r3, h = train_full_pass(m_r3, train_days, EPOCHS_FT, lr_ft, f"{family_name}_R3_finetune", eval_bank=val_days)
    histories += h
    models["R3 pretrain+finetune"] = copy.deepcopy(m_r3).cpu()
    res = evaluate_model(m_r3, test_days, tag=f"{family_name}_R3_pretrain_finetune")
    results.append(dict(model=family_name, regime="R3 pretrain+finetune", **res))

    save_history(histories, f"nb04_{family_name}_training_curves.parquet")
    state_bundle = {
        regime: model.state_dict()
        for regime, model in models.items()
    }
    torch.save(state_bundle, RUN_DIR / f"nb04_{family_name}_state_dicts.pt")
    print("written:", RUN_DIR / f"nb04_{family_name}_state_dicts.pt")
    return histories, results, models


all_histories, all_results, trained_models = [], [], {}

deep_hist, deep_res, deep_models = run_family(DeepSetONet, "deeponet", lr_pre=3e-3, lr_real=3e-3, lr_ft=1e-3)
all_histories += deep_hist
all_results += deep_res
trained_models["deeponet"] = deep_models

gno_hist, gno_res, gno_models = run_family(InterpGNO, "gno", lr_pre=1e-3, lr_real=1e-3, lr_ft=5e-4)
all_histories += gno_hist
all_results += gno_res
trained_models["gno"] = gno_models

summary_path = RUN_DIR / "nb04_operator_summary.parquet"
hist_path = RUN_DIR / "nb04_all_training_curves.parquet"
pl.DataFrame(all_results).write_parquet(summary_path)
pl.DataFrame(all_histories).write_parquet(hist_path)
print("written:", summary_path)
print("written:", hist_path)
pl.DataFrame(all_results)


MODEL FAMILY: deeponet

R2: synthetic-only
params: 67009
[deeponet_R2_pretrain] epoch   1/25 full-pass loss=0.2467 | val rmse=6.157 vol pts | elapsed=2s
[deeponet_R2_pretrain] epoch   5/25 full-pass loss=0.0801 | val rmse=4.835 vol pts | elapsed=8s
[deeponet_R2_pretrain] epoch  10/25 full-pass loss=0.0640 | val rmse=3.810 vol pts | elapsed=15s
[deeponet_R2_pretrain] epoch  15/25 full-pass loss=0.0621 | val rmse=3.290 vol pts | elapsed=23s
[deeponet_R2_pretrain] epoch  20/25 full-pass loss=0.0557 | val rmse=2.781 vol pts | elapsed=31s
[deeponet_R2_pretrain] epoch  25/25 full-pass loss=0.0562 | val rmse=3.751 vol pts | elapsed=38s

R1: real-only
params: 67009
[deeponet_R1_real] epoch   1/25 full-pass loss=1.0105 | val rmse=26.127 vol pts | elapsed=3s
[deeponet_R1_real] epoch   5/25 full-pass loss=1.0000 | val rmse=26.127 vol pts | elapsed=16s
[deeponet_R1_real] epoch  10/25 full-pass loss=1.0000 | val rmse=26.127 vol pts | elapsed=32s
[deeponet_R1_real] epoch  15/25 full-pass loss=1.000

model,regime,tag,n_days,rmse_volpts,rmse_sd_volpts,mae_volpts,rel_rmse,min_g,butterfly_viol_pct,calendar_viol_pct
str,str,str,i64,f64,f64,f64,f64,f64,f64,f64
"""deeponet""","""R2 synth-only""","""deeponet_R2_synth_only""",386,6.269052,0.676176,4.193534,0.204052,0.135662,0.0,0.0
"""deeponet""","""R1 real-only""","""deeponet_R1_real_only""",386,22.255183,2.051354,20.470381,0.999994,0.999517,0.0,0.0
"""deeponet""","""R3 pretrain+finetune""","""deeponet_R3_pretrain_finetune""",386,2.269412,0.44639,1.386762,0.103855,0.007111,0.0,0.0
"""gno""","""R2 synth-only""","""gno_R2_synth_only""",386,4.577445,0.590388,2.749908,0.16768,-2.406668,0.637594,0.030911
"""gno""","""R1 real-only""","""gno_R1_real_only""",386,1.815176,0.403936,1.052279,0.063092,-0.579565,0.083477,0.0
"""gno""","""R3 pretrain+finetune""","""gno_R3_pretrain_finetune""",386,1.960501,0.43849,1.100697,0.066495,-0.602655,0.112263,0.0


## 6. Analysis-ready outputs: per-day metrics, invariance, regimes

The next cells create the artifacts that can feed the paper directly:

- per-test-day RMSE and arbitrage audit for each model/regime;
- invariance-to-subsampling diagnostics;
- simple market-regime grouping by median IV level;
- comparison plots.

In [8]:
@torch.no_grad()
def per_day_metrics(model, model_name, regime, days):
    rows = []
    model = model.to(DEVICE)
    for surf in days:
        pred = predict_np(model, surf, surf["k"], surf["tau"])
        err = pred - surf["iv"]
        V = predict_np(model, surf, KG_FLAT, TG_FLAT).reshape(GRID_NT, GRID_NK)
        audit = fd_audit_from_grid(V)
        rows.append(dict(
            model=model_name,
            regime=regime,
            date=str(surf["date"]),
            n_quotes=int(len(surf["k"])),
            iv_level=float(np.median(surf["iv"])),
            rmse_volpts=float(np.sqrt(np.mean(err ** 2)) * 100),
            mae_volpts=float(np.mean(np.abs(err)) * 100),
            rel_rmse=float(np.sqrt(np.mean((err / surf["iv"]) ** 2))),
            **audit,
        ))
    return rows


per_day_rows = []
for model_name, regimes in trained_models.items():
    for regime, model in regimes.items():
        per_day_rows.extend(per_day_metrics(model, model_name, regime, test_days))

per_day = pl.DataFrame(per_day_rows)
q1, q2 = per_day.select(pl.col("iv_level").quantile(0.33)).item(), per_day.select(pl.col("iv_level").quantile(0.66)).item()
per_day = per_day.with_columns(
    pl.when(pl.col("iv_level") <= q1).then(pl.lit("low_iv"))
    .when(pl.col("iv_level") <= q2).then(pl.lit("mid_iv"))
    .otherwise(pl.lit("high_iv"))
    .alias("regime_bucket")
)
per_day_path = RUN_DIR / "nb04_operator_per_day.parquet"
per_day.write_parquet(per_day_path)
print("written:", per_day_path)
per_day.head()

written: data/clean/nb04_operator_run/nb04_operator_per_day.parquet


model,regime,date,n_quotes,iv_level,rmse_volpts,mae_volpts,rel_rmse,min_g,butterfly_viol_pct,calendar_viol_pct,regime_bucket
str,str,str,i64,f64,f64,f64,f64,f64,f64,f64,str
"""deeponet""","""R2 synth-only""","""2024-02-15""",2500,0.16339,6.818158,4.497225,0.222277,0.168282,0.0,0.0,"""low_iv"""
"""deeponet""","""R2 synth-only""","""2024-02-16""",2500,0.1601125,6.638223,4.36223,0.21587,0.16384,0.0,0.0,"""low_iv"""
"""deeponet""","""R2 synth-only""","""2024-02-20""",2500,0.1667355,6.45148,4.17972,0.202522,0.174517,0.0,0.0,"""mid_iv"""
"""deeponet""","""R2 synth-only""","""2024-02-21""",2500,0.1630105,6.640464,4.19439,0.203756,0.169806,0.0,0.0,"""low_iv"""
"""deeponet""","""R2 synth-only""","""2024-02-22""",2500,0.1692395,7.198536,4.650941,0.221417,0.188887,0.0,0.0,"""mid_iv"""


In [9]:
def subsurf(surf, frac, r):
    n = len(surf["k"])
    m = max(8, int(frac * n))
    idx = np.sort(r.choice(n, size=min(m, n), replace=False))
    return dict(date=surf["date"], k=surf["k"][idx], tau=surf["tau"][idx], iv=surf["iv"][idx], kind=surf.get("kind", "real"))


@torch.no_grad()
def invariance_rows(model, model_name, regime, days, fracs=(1.0, 0.75, 0.50, 0.25), n_days=10):
    r = np.random.default_rng(SEED + 99)
    rows = []
    model = model.to(DEVICE)
    for surf in days[:min(n_days, len(days))]:
        base = predict_np(model, surf, KG_FLAT, TG_FLAT).reshape(GRID_NT, GRID_NK)
        for frac in fracs:
            s2 = subsurf(surf, frac, r)
            V = predict_np(model, s2, KG_FLAT, TG_FLAT).reshape(GRID_NT, GRID_NK)
            rows.append(dict(
                model=model_name, regime=regime, date=str(surf["date"]),
                input_frac=float(frac), n_input=int(len(s2["k"])),
                mean_abs_surface_shift_volpts=float(np.mean(np.abs(V - base)) * 100),
                max_abs_surface_shift_volpts=float(np.max(np.abs(V - base)) * 100),
            ))
    return rows


inv = []
for model_name, regimes in trained_models.items():
    for regime, model in regimes.items():
        inv.extend(invariance_rows(model, model_name, regime, test_days))

inv_df = pl.DataFrame(inv)
inv_path = RUN_DIR / "nb04_operator_invariance.parquet"
inv_df.write_parquet(inv_path)
print("written:", inv_path)
inv_df.head()

written: data/clean/nb04_operator_run/nb04_operator_invariance.parquet


model,regime,date,input_frac,n_input,mean_abs_surface_shift_volpts,max_abs_surface_shift_volpts
str,str,str,f64,i64,f64,f64
"""deeponet""","""R2 synth-only""","""2024-02-15""",1.0,2500,0.0,0.0
"""deeponet""","""R2 synth-only""","""2024-02-15""",0.75,1875,0.086606,0.275868
"""deeponet""","""R2 synth-only""","""2024-02-15""",0.5,1250,0.051055,0.098848
"""deeponet""","""R2 synth-only""","""2024-02-15""",0.25,625,0.1332,0.247147
"""deeponet""","""R2 synth-only""","""2024-02-16""",1.0,2500,0.0,0.0


## 7. Tables and figures for the paper

In [10]:
summary = pl.read_parquet(RUN_DIR / "nb04_operator_summary.parquet")
print(summary.sort(["model", "rmse_volpts"]))

fig = go.Figure()
for model_name in summary["model"].unique().to_list():
    sub = summary.filter(pl.col("model") == model_name).sort("regime")
    fig.add_trace(go.Bar(
        name=model_name,
        x=sub["regime"].to_list(),
        y=sub["rmse_volpts"].to_list(),
        error_y=dict(array=sub["rmse_sd_volpts"].to_list()),
    ))
fig.update_layout(
    width=950, height=460, barmode="group",
    title="NB04 test performance: operator families and transfer regimes",
    yaxis_title="Test IV RMSE (vol points)",
    xaxis_title="training regime",
)
fig.show()

shape: (6, 11)
┌──────────┬────────────┬────────────┬────────┬───┬──────────┬───────────┬────────────┬────────────┐
│ model    ┆ regime     ┆ tag        ┆ n_days ┆ … ┆ rel_rmse ┆ min_g     ┆ butterfly_ ┆ calendar_v │
│ ---      ┆ ---        ┆ ---        ┆ ---    ┆   ┆ ---      ┆ ---       ┆ viol_pct   ┆ iol_pct    │
│ str      ┆ str        ┆ str        ┆ i64    ┆   ┆ f64      ┆ f64       ┆ ---        ┆ ---        │
│          ┆            ┆            ┆        ┆   ┆          ┆           ┆ f64        ┆ f64        │
╞══════════╪════════════╪════════════╪════════╪═══╪══════════╪═══════════╪════════════╪════════════╡
│ deeponet ┆ R3 pretrai ┆ deeponet_R ┆ 386    ┆ … ┆ 0.103855 ┆ 0.007111  ┆ 0.0        ┆ 0.0        │
│          ┆ n+finetune ┆ 3_pretrain ┆        ┆   ┆          ┆           ┆            ┆            │
│          ┆            ┆ _finetune  ┆        ┆   ┆          ┆           ┆            ┆            │
│ deeponet ┆ R2         ┆ deeponet_R ┆ 386    ┆ … ┆ 0.204052 ┆ 0.135662  ┆ 0

In [11]:
per_day = pl.read_parquet(RUN_DIR / "nb04_operator_per_day.parquet")
regime_table = (
    per_day.group_by(["model", "regime", "regime_bucket"])
    .agg(
        pl.col("rmse_volpts").mean().alias("mean_rmse_volpts"),
        pl.col("butterfly_viol_pct").mean().alias("mean_bfly_viol_pct"),
        pl.col("calendar_viol_pct").mean().alias("mean_cal_viol_pct"),
        pl.len().alias("n_days"),
    )
    .sort(["model", "regime", "regime_bucket"])
)
regime_table_path = RUN_DIR / "nb04_operator_regime_table.parquet"
regime_table.write_parquet(regime_table_path)
print("written:", regime_table_path)
print(regime_table)

written: data/clean/nb04_operator_run/nb04_operator_regime_table.parquet
shape: (18, 7)
┌──────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────┬────────┐
│ model    ┆ regime        ┆ regime_bucket ┆ mean_rmse_vol ┆ mean_bfly_vio ┆ mean_cal_vio ┆ n_days │
│ ---      ┆ ---           ┆ ---           ┆ pts           ┆ l_pct         ┆ l_pct        ┆ ---    │
│ str      ┆ str           ┆ str           ┆ ---           ┆ ---           ┆ ---          ┆ u32    │
│          ┆               ┆               ┆ f64           ┆ f64           ┆ f64          ┆        │
╞══════════╪═══════════════╪═══════════════╪═══════════════╪═══════════════╪══════════════╪════════╡
│ deeponet ┆ R1 real-only  ┆ high_iv       ┆ 24.302432     ┆ 0.0           ┆ 0.0          ┆ 131    │
│ deeponet ┆ R1 real-only  ┆ low_iv        ┆ 20.524628     ┆ 0.0           ┆ 0.0          ┆ 128    │
│ deeponet ┆ R1 real-only  ┆ mid_iv        ┆ 21.887636     ┆ 0.0           ┆ 0.0          ┆ 127    │
│ d

In [12]:
inv_df = pl.read_parquet(RUN_DIR / "nb04_operator_invariance.parquet")
inv_mean = (
    inv_df.group_by(["model", "regime", "input_frac"])
    .agg(pl.col("mean_abs_surface_shift_volpts").mean().alias("shift"))
    .sort(["model", "regime", "input_frac"])
)

fig = go.Figure()
for model_name in inv_mean["model"].unique().to_list():
    for regime in inv_mean.filter(pl.col("model") == model_name)["regime"].unique().to_list():
        sub = inv_mean.filter((pl.col("model") == model_name) & (pl.col("regime") == regime)).sort("input_frac")
        fig.add_trace(go.Scatter(
            x=(sub["input_frac"] * 100).to_list(),
            y=sub["shift"].to_list(),
            mode="lines+markers",
            name=f"{model_name} | {regime}",
        ))
fig.update_layout(
    width=950, height=470,
    title="Discretization invariance: dense surface shift under input subsampling",
    xaxis_title="% of input quotes retained",
    yaxis_title="mean absolute surface shift (vol points)",
)
fig.update_xaxes(autorange="reversed")
fig.show()

## 8. Visual audit on one unseen test day

In [13]:
best_row = summary.sort("rmse_volpts").row(0, named=True)
best_model_name, best_regime = best_row["model"], best_row["regime"]
model = trained_models[best_model_name][best_regime].to(DEVICE)
surf = test_days[-1]

fig = go.Figure()
taus = np.unique(np.round(surf["tau"], 3))
sel = taus[np.linspace(0, len(taus) - 1, min(5, len(taus))).round().astype(int)]
colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"]
for tau, color in zip(sel, colors):
    m = np.isclose(np.round(surf["tau"], 3), tau)
    if m.sum() < 4:
        continue
    kk = np.linspace(surf["k"][m].min(), surf["k"][m].max(), 150)
    vv = predict_np(model, surf, kk, np.full_like(kk, tau))
    fig.add_trace(go.Scatter(x=surf["k"][m], y=surf["iv"][m], mode="markers", marker=dict(color=color, size=6), name=f"{tau*365:.0f}d quotes"))
    fig.add_trace(go.Scatter(x=kk, y=vv, line=dict(color=color), showlegend=False))
fig.update_layout(
    width=920, height=450,
    title=f"Best NB04 operator on unseen day {surf['date']}: {best_model_name}, {best_regime}",
    xaxis_title="log-moneyness k", yaxis_title="IV",
)
fig.show()

V = predict_np(model, surf, KG_FLAT, TG_FLAT).reshape(GRID_NT, GRID_NK)
fig = go.Figure(go.Surface(x=KG, y=TG, z=V, colorscale="Viridis", colorbar=dict(title="IV")))
fig.update_layout(width=780, height=520, title="Operator-smoothed IV surface", scene=dict(xaxis_title="k", yaxis_title="tau", zaxis_title="IV"))
fig.show()

W = V ** 2 * TTG
Wk = (W[:, 2:] - W[:, :-2]) / (2 * DK)
Wkk = (W[:, 2:] - 2 * W[:, 1:-1] + W[:, :-2]) / (DK ** 2)
G = durrleman_g_np(KKG[:, 1:-1], W[:, 1:-1], Wk, Wkk)
fig = go.Figure(go.Heatmap(z=G, x=KG[1:-1], y=TG, zmid=0, colorscale="RdBu", colorbar=dict(title="g")))
fig.update_layout(width=760, height=420, title="Butterfly audit: Durrleman g(k,tau)", xaxis_title="k", yaxis_title="tau")
fig.show()

## 9. Reproducibility record

In [14]:
run_config = dict(
    seed=SEED,
    device=str(DEVICE),
    real_parquet=str(REAL_PARQUET),
    n_real_days=len(real_bank),
    n_synth=N_SYNTH,
    train_days=len(train_days),
    val_days=len(val_days),
    test_days=len(test_days),
    epochs_pre=EPOCHS_PRE,
    epochs_r1=EPOCHS_R1,
    epochs_ft=EPOCHS_FT,
    max_quotes_load=MAX_QUOTES_LOAD,
    fit_points_deep=FIT_POINTS_DEEP,
    input_points_deep=INPUT_POINTS_DEEP,
    fit_points_gno=FIT_POINTS_GNO,
    input_points_gno=INPUT_POINTS_GNO,
    batch_deep=BATCH_DEEP,
    batch_gno=BATCH_GNO,
    latent=LATENT,
    enc_h=ENC_H,
    trunk_h=TRUNK_H,
    gno_channels=GNO_CHANNELS,
    gno_layers=GNO_LAYERS,
    gno_k=GNO_K,
    grid_nk=GRID_NK,
    grid_nt=GRID_NT,
    lambdas=LAMBDAS,
)
config_path = RUN_DIR / "nb04_run_config.json"
config_path.write_text(json.dumps(run_config, indent=2))
print("written:", config_path)
print(json.dumps(run_config, indent=2))

written: data/clean/nb04_operator_run/nb04_run_config.json
{
  "seed": 0,
  "device": "cpu",
  "real_parquet": "data/clean/option_prices_clean.parquet",
  "n_real_days": 1926,
  "n_synth": 1000,
  "train_days": 1155,
  "val_days": 385,
  "test_days": 386,
  "epochs_pre": 25,
  "epochs_r1": 25,
  "epochs_ft": 12,
  "max_quotes_load": 2500,
  "fit_points_deep": 512,
  "input_points_deep": 900,
  "fit_points_gno": 160,
  "input_points_gno": 220,
  "batch_deep": 8,
  "batch_gno": 1,
  "latent": 64,
  "enc_h": 128,
  "trunk_h": 128,
  "gno_channels": 8,
  "gno_layers": 2,
  "gno_k": 16,
  "grid_nk": 22,
  "grid_nt": 9,
  "lambdas": {
    "fit": 1.0,
    "but": 10.0,
    "cal": 10.0,
    "reg_t": 0.01,
    "reg_k": 0.01
  }
}


## 10. Research interpretation checklist

Use the generated Parquet files for the paper:

- `nb04_operator_summary.parquet`: headline test metrics by model and transfer regime.
- `nb04_operator_per_day.parquet`: per-day errors and arbitrage audits.
- `nb04_operator_invariance.parquet`: discretization-invariance test.
- `nb04_operator_regime_table.parquet`: performance by low/mid/high IV regime.
- `nb04_all_training_curves.parquet`: optimization curves.
- `nb04_deeponet_models.pt`, `nb04_gno_models.pt`: trained PyTorch models.

The main research claims to inspect:

1. **Low-data viability:** does R1 real-only generalize to later test days?
2. **Synthetic transfer:** does R2 beat or approach R1 despite seeing no real training days?
3. **Pretraining value:** does R3 improve over R1, especially in high-IV regimes?
4. **Operator property:** are surfaces stable under 75/50/25% input subsampling?
5. **Arbitrage control:** does lower RMSE come with acceptable butterfly/calendar diagnostics?